# Lecture 9: Model Evaluation and Selection
**BITE 485 | Tharaka University**

---

## Learning Outcomes
1. Construct and interpret a confusion matrix
2. Calculate Precision, Recall, F1-Score, and AUC-ROC
3. Evaluate a model on an imbalanced dataset
4. Apply SMOTE and class weighting for imbalance
5. Tune hyperparameters with GridSearchCV

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                              classification_report, roc_auc_score,
                              roc_curve, precision_recall_curve)
try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
except ImportError:
    print('imbalanced-learn not installed. Install with: pip install imbalanced-learn')
    SMOTE_AVAILABLE = False
print('Ready.')

## 9.1 Simulating a Fraud Detection Dataset (Imbalanced)

In [ ]:
np.random.seed(42)
X, y = make_classification(
    n_samples=5000, n_features=15, n_informative=8,
    n_redundant=3, weights=[0.97, 0.03],  # 3% fraud
    random_state=42
)
print(f'Dataset shape: {X.shape}')
print(f'Fraud rate: {y.mean()*100:.1f}%')
print(f'Fraud cases: {y.sum()} | Legitimate: {(y==0).sum()}')

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_te)
print(f'Train fraud: {y_tr.sum()} | Test fraud: {y_te.sum()}')

## 9.2 Baseline Model — Illustrating the Accuracy Paradox

In [ ]:
# Naive classifier: always predict 'not fraud'
y_naive = np.zeros(len(y_te), dtype=int)
from sklearn.metrics import accuracy_score
print(f'Naive classifier accuracy: {accuracy_score(y_te, y_naive)*100:.1f}%')
print('This is misleading! It never detects a single fraud case.')

## 9.3 Confusion Matrix and Metrics

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_tr_sc, y_tr)
y_pred = rf.predict(X_te_sc)
y_proba = rf.predict_proba(X_te_sc)[:, 1]

print('=== Confusion Matrix ===')
cm = confusion_matrix(y_te, y_pred)
print(cm)
print('Format: [[TN FP] [FN TP]]')

tn, fp, fn, tp = cm.ravel()
print(f'\nTP={tp}  TN={tn}  FP={fp}  FN={fn}')
print(f'\nPrecision: {tp/(tp+fp):.4f}  (of predicted fraud, how many were fraud?)')
print(f'Recall:    {tp/(tp+fn):.4f}  (of all fraud, how many did we catch?)')
print(f'F1-Score:  {2*tp/(2*tp+fp+fn):.4f}')
print(f'Accuracy:  {(tp+tn)/(tp+tn+fp+fn):.4f}')
print(f'AUC-ROC:   {roc_auc_score(y_te, y_proba):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix visualisation
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Legitimate', 'Fraud'])
disp.plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, _ = roc_curve(y_te, y_proba)
auc = roc_auc_score(y_te, y_proba)
axes[1].plot(fpr, tpr, 'b-', label=f'ROC (AUC={auc:.3f})')
axes[1].plot([0,1],[0,1],'r--', label='Random (AUC=0.5)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate (Recall)')
axes[1].set_title('ROC Curve')
axes[1].legend()
plt.tight_layout()
plt.show()

## 9.4 Handling Class Imbalance

In [ ]:
results = []
for name, clf in [
    ('RF — No weighting',       RandomForestClassifier(100, random_state=42)),
    ('RF — class_weight balanced', RandomForestClassifier(100, random_state=42, class_weight='balanced')),
    ('LR — class_weight balanced', LogisticRegression(class_weight='balanced', max_iter=1000))
]:
    clf.fit(X_tr_sc, y_tr)
    pr = clf.predict(X_te_sc)
    prob = clf.predict_proba(X_te_sc)[:, 1]
    tn2,fp2,fn2,tp2 = confusion_matrix(y_te, pr).ravel()
    results.append({'Model': name,
                    'Recall (fraud)': f'{tp2/(tp2+fn2):.3f}',
                    'Precision (fraud)': f'{tp2/(tp2+fp2):.3f}',
                    'F1': f'{2*tp2/(2*tp2+fp2+fn2):.3f}',
                    'AUC-ROC': f'{roc_auc_score(y_te, prob):.3f}'})

if SMOTE_AVAILABLE:
    sm = SMOTE(random_state=42)
    X_sm, y_sm = sm.fit_resample(X_tr_sc, y_tr)
    rf_sm = RandomForestClassifier(100, random_state=42)
    rf_sm.fit(X_sm, y_sm)
    pr = rf_sm.predict(X_te_sc)
    prob = rf_sm.predict_proba(X_te_sc)[:, 1]
    tn2,fp2,fn2,tp2 = confusion_matrix(y_te, pr).ravel()
    results.append({'Model': 'RF — SMOTE oversampling',
                    'Recall (fraud)': f'{tp2/(tp2+fn2):.3f}',
                    'Precision (fraud)': f'{tp2/(tp2+fp2):.3f}',
                    'F1': f'{2*tp2/(2*tp2+fp2+fn2):.3f}',
                    'AUC-ROC': f'{roc_auc_score(y_te, prob):.3f}'})

pd.DataFrame(results)

## 9.5 Hyperparameter Tuning with GridSearchCV

In [ ]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth':    [5, 10, None],
    'min_samples_split': [2, 10]
}

gs = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid, cv=3, scoring='f1', n_jobs=-1, verbose=0
)
gs.fit(X_tr_sc, y_tr)

print(f'Best parameters: {gs.best_params_}')
print(f'Best CV F1-Score: {gs.best_score_:.4f}')

best_model = gs.best_estimator_
y_best = best_model.predict(X_te_sc)
print(f'\nTest F1: {2*sum((y_best==1)&(y_te==1))/(2*sum((y_best==1)&(y_te==1))+sum((y_best==1)&(y_te==0))+sum((y_best==0)&(y_te==1))):.4f}')

### Exercise

Using the fraud detection dataset:

1. Plot the Precision-Recall curve for the best model from GridSearchCV.
2. Find the classification threshold that maximises F1-Score (hint: iterate over threshold values from 0.1 to 0.9 and compute F1 at each).
3. Compare performance at the default threshold (0.5) vs your optimal threshold.
4. Write a one-paragraph summary of which model and threshold you would recommend for deployment, explaining the business trade-off between Precision and Recall in a fraud context.

In [ ]:
# Your code here


---
*BITE 485 Data Science | Tharaka University | kevin.tuei@tharaka.ac.ke*